# Track a watershed through a drought

**Terrain intermediate · Python project · USGS Water Data**

Follow daily streamflow at multiple USGS gages across Southern California before, during, and after recent drought years.

---

## Learning objectives

By the end, you should be able to:

- Query a public hydrology API for multiple sites at once
- Reshape long data into comparisons across gages and years
- Visualize drought periods on a hydrograph
- Interpret flow changes without confusing causation

**Time:** ~2 hr · **Dataset:** [USGS Water Data explainer](../explainer.html?id=usgs-water)

---

## For mentors

- **Prerequisites:** Completed at least one beginner Python project.
- **Vocabulary:** *Discharge* (cfs) = volume of water passing the gage per second. Higher = wetter moment.
- **Watch for:** Students comparing cfs across gages without noting watershed size — address in takeaway.
- **No API key** — USGS NWIS is open. If download fails, check site IDs or date range.

**Suggested flow:** Read each section aloud briefly → student runs the cell → use the *Mentor check-in* question before moving on. Do not skip the final takeaway.

---


**Goal:** Import plotting tools and set visual defaults.

**Concept:** We'll use `matplotlib` for a multi-panel 'dashboard' — one subplot per river gage.

**Run the cell below**, then compare your output to what we expect.

**You should see:** No output.

**Mentor check-in:** Ask: *What is streamflow/discharge measuring — rainfall, river width, or something else?*


In [ ]:
import pandas as pd
import requests
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.rcParams.update({"figure.facecolor": "white", "axes.spines.top": False, "axes.spines.right": False})

**Goal:** Choose gages and the study period.

**Concept:** Each **site ID** is an 8-digit USGS gage. We shade drought years to help the eye — drought is complex, but these years were notably dry in California.

**Run the cell below**, then compare your output to what we expect.

**You should see:** No output — review the `SITES` dictionary together.

**Mentor check-in:** Ask: *Why three gages instead of one?* (Same climate stress, different watershed response.)


In [ ]:
SITES = {
    "11098000": "Arroyo Seco nr Pasadena",
    "11087020": "San Gabriel River ab Whittier Narrows Dam",
    "11113000": "Sespe Creek nr Fillmore",
}

START_DATE = "2015-01-01"
END_DATE = "2024-12-31"
DROUGHT_YEARS = [2015, 2016, 2021, 2022]  # shaded bands on the chart

**Goal:** Download daily mean discharge (parameter code 00060).

**Concept:** The API returns JSON; we parse it into a tidy table: date, site, cfs. Missing values become `NaN` and are dropped.

**Run the cell below**, then compare your output to what we expect.

**You should see:** Thousands of rows printed; `head()` shows date, site name, cfs.

**Mentor check-in:** Ask: *What does cfs stand for, and why might it spike after rain?*

**If something breaks:** HTTP error → verify internet. Empty table → check site IDs on waterdata.usgs.gov.


In [ ]:
def fetch_usgs_daily(site_ids, start, end, param="00060"):
    url = "https://waterservices.usgs.gov/nwis/dv/"
    params = {
        "format": "json",
        "sites": ",".join(site_ids),
        "parameterCd": param,
        "startDT": start,
        "endDT": end,
        "siteStatus": "all",
    }
    resp = requests.get(url, params=params, timeout=60)
    resp.raise_for_status()
    payload = resp.json()

    rows = []
    for series in payload.get("value", {}).get("timeSeries", []):
        site = series["sourceInfo"]["siteCode"][0]["value"]
        name = series["sourceInfo"]["siteName"]
        for v in series["values"][0]["value"]:
            val = v["value"]
            rows.append({
                "site_id": site,
                "site_name": name,
                "date": v["dateTime"][:10],
                "cfs": float(val) if val not in ("-999999", "") else None,
            })
    df = pd.DataFrame(rows)
    df["date"] = pd.to_datetime(df["date"])
    return df

flow = fetch_usgs_daily(list(SITES.keys()), START_DATE, END_DATE)
flow = flow.dropna(subset=["cfs"])
print(f"Downloaded {len(flow):,} daily records across {flow['site_id'].nunique()} sites")
flow.head()

**Goal:** Summarize mean flow by year — drought signal in numbers.

**Concept:** A pivot table lets you compare the same year across rivers side by side.

**Run the cell below**, then compare your output to what we expect.

**You should see:** Table with years as rows and each gage as a column (mean cfs).

**Mentor check-in:** Ask: *Which year looks lowest for each gage? Does it match the shaded drought years?*


In [ ]:
annual = (
    flow.assign(year=flow["date"].dt.year)
        .groupby(["site_id", "site_name", "year"], as_index=False)["cfs"]
        .mean()
        .rename(columns={"cfs": "mean_cfs"})
)
annual.pivot(index="year", columns="site_name", values="mean_cfs").round(1)

**Goal:** Build the multi-site hydrograph dashboard.

**Concept:** Orange bands = drought years we flagged. Trace the green line in each panel — where does it dip?

**Run the cell below**, then compare your output to what we expect.

**You should see:** Three stacked time-series plots with shaded drought years.

**Mentor check-in:** Ask: *Did all three gages recover the same way after 2022?*


In [ ]:
fig, axes = plt.subplots(len(SITES), 1, figsize=(10, 3.2 * len(SITES)), sharex=True)
if len(SITES) == 1:
    axes = [axes]

for ax, (site_id, label) in zip(axes, SITES.items()):
    sub = flow[flow["site_id"] == site_id].sort_values("date")
    ax.plot(sub["date"], sub["cfs"], color="#2F6B4F", linewidth=0.9)
    for yr in DROUGHT_YEARS:
        ax.axvspan(pd.Timestamp(f"{yr}-01-01"), pd.Timestamp(f"{yr}-12-31"),
                   color="#D98E5A", alpha=0.12)
    ax.set_ylabel("cfs")
    ax.set_title(label, loc="left", fontsize=12, fontweight="bold")
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

axes[-1].set_xlabel("Date")
fig.suptitle("Southern California streamflow through drought years (USGS daily discharge)",
             fontsize=14, fontweight="bold", y=1.01)
fig.tight_layout()
plt.show()

---

## Final takeaway

**Mentor guides discussion:**

1. Which gage shows the largest drop during shaded drought years?
2. Did flows recover equally after 2022?
3. Why shouldn't you compare raw cfs across gages without context about watershed size?

**Extension:** Add NOAA precipitation for the same years and discuss rain vs. flow.

**Next:** [USGS explainer](../explainer.html?id=usgs-water) · [Terrain datasets](../datasets.html)
